# Full System Integration — Sarah (System 1) + Russell (System 2) + Youssef (System 3)

End-to-end pipeline: a claim goes through System 1's classifier, System 2's RAG fact-verification engine, then System 3's verdict fusion — producing one final, reasoned answer. This notebook is for **integration testing only**; System 2's own development/calibration work lives in `system2_local_clean.ipynb`.

## 1. Setup — System 2's models and pipeline (reused from `system2_local_clean.ipynb`)

## 1. Setup — models, data, and helper functions
Run this whole section once, top to bottom. Loads all models (cached locally, fast after first run) and connects to the Chroma vector DB (Bucket A + B already built on disk — these cells detect that and skip re-ingestion).

In [1]:
import os, re, subprocess
import pandas as pd
import numpy as np
import torch
import chromadb
from transformers import pipeline, AutoTokenizer, AutoModel
from camel_tools.utils.normalize import normalize_alef_ar, normalize_unicode

# Install rank-bm25 if not present
subprocess.run(["pip", "install", "rank-bm25", "-q"])
from rank_bm25 import BM25Okapi

# ── Paths ────────────────────────────────────────────────────────────────────
BASE   = "/Users/russelltamer/Desktop/system 2 RAG"
PROPS  = f"{BASE}/df_propositions_all.csv"      # merged: original 8,376 + v2 5,808 + SANAD 6,656 = 20,687
CHROMA = f"{BASE}/chroma_db"

for name, path in [("Propositions CSV", PROPS)]:
    status = "✅" if os.path.exists(path) else "❌ MISSING"
    print(f"{status}  {name}: {path}")

print("\n✅ All imports OK")

✅  Propositions CSV: /Users/russelltamer/Desktop/system 2 RAG/df_propositions_all.csv

✅ All imports OK


In [2]:
# ── Dialect classifier (CAMeL) ───────────────────────────────────────────────
print("Loading dialect classifier...")
dialect_classifier = pipeline(
    "text-classification",
    model="CAMeL-Lab/bert-base-arabic-camelbert-mix-did-madar-corpus6"
)
print("✅ Dialect classifier loaded  (label 'CAI' = Egyptian Arabic)")

# ── multilingual-e5-large ─────────────────────────────────────────────────────
# Retrieval-optimized: trained on (query, passage) pairs.
# Requires prefix: "query: " for queries, "passage: " for documents.
# v1 used AraBERT (CLS token, 768-dim) — task mismatch caused P@1=0.415.
# E5-large is purpose-built for retrieval → much stronger semantic matching.
print("\nLoading multilingual-e5-large...")
E5_MODEL_NAME = "intfloat/multilingual-e5-large"
e5_tokenizer  = AutoTokenizer.from_pretrained(E5_MODEL_NAME)
e5_model      = AutoModel.from_pretrained(E5_MODEL_NAME)
e5_model.eval()
print("✅ E5-large loaded  (1024-dim, mean pooling)")

def _mean_pool(token_embeds, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(token_embeds.size()).float()
    return (token_embeds * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

def get_embedding(text: str, prefix: str = "query") -> np.ndarray:
    inp = e5_tokenizer(f"{prefix}: {text}", return_tensors="pt",
                       truncation=True, max_length=512, padding=True)
    with torch.no_grad():
        out = e5_model(**inp)
    vec = _mean_pool(out.last_hidden_state, inp['attention_mask'])
    vec = torch.nn.functional.normalize(vec, p=2, dim=1)
    return vec.squeeze().numpy()

def get_embeddings_batch(texts: list, batch_size: int = 16,
                          prefix: str = "passage") -> np.ndarray:
    prefixed = [f"{prefix}: {t}" for t in texts]
    all_vecs = []
    for i in range(0, len(prefixed), batch_size):
        batch = prefixed[i:i+batch_size]
        inp = e5_tokenizer(batch, return_tensors="pt", truncation=True,
                           max_length=512, padding=True)
        with torch.no_grad():
            out = e5_model(**inp)
        vecs = _mean_pool(out.last_hidden_state, inp['attention_mask'])
        vecs = torch.nn.functional.normalize(vecs, p=2, dim=1)
        all_vecs.append(vecs.numpy())
        if (i // batch_size) % 10 == 0:
            print(f"  Embedded {i+len(batch)}/{len(texts)}")
    return np.vstack(all_vecs)

vec = get_embedding("اختبار")
print(f"✅ Embedding shape: {vec.shape}")  # expect (1024,)

# ── NER model ─────────────────────────────────────────────────────────────────
print("\nLoading NER model...")
ner_pipeline_ar = pipeline(
    "ner",
    model="CAMeL-Lab/bert-base-arabic-camelbert-mix-ner",
    aggregation_strategy="simple",
    device=-1
)
print("✅ NER model loaded")

Loading dialect classifier...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Dialect classifier loaded  (label 'CAI' = Egyptian Arabic)

Loading multilingual-e5-large...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ E5-large loaded  (1024-dim, mean pooling)
✅ Embedding shape: (1024,)

Loading NER model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-mix-ner
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ NER model loaded


In [3]:
# ── AraFacts 2 ───────────────────────────────────────────────────────────────
df_arafacts = pd.read_csv(f"{BASE}/AraFacts 2.csv")
print(f"AraFacts shape: {df_arafacts.shape}")
print(f"Columns: {df_arafacts.columns.tolist()}")
print(df_arafacts['normalized_label'].value_counts())

# Filter descriptions for Bucket B (non-empty, ≥ 40 chars)
df_desc = df_arafacts[
    df_arafacts['description'].notna() &
    (df_arafacts['description'].str.len() > 40)
].copy().reset_index(drop=True)
print(f"\nAraFacts descriptions available for Bucket B: {len(df_desc)}")


AraFacts shape: (18598, 14)
Columns: ['ClaimID', 'claim', 'description', 'source', 'date', 'source_label', 'normalized_label', 'source_category', 'normalized_category', 'source_url', 'claim_urls', 'evidence_urls', 'evidence_urls.1', 'claim_type']
normalized_label
FALSE           10901
Partly-false     6862
TRUE              426
Sarcasm           181
Partly-False      172
Unverifiable       56
Name: count, dtype: int64

AraFacts descriptions available for Bucket B: 18127


**Claude API setup** — needed for decontextualizing the top Bucket B evidence shown to users. Must run *before* the main pipeline function below (this was previously positioned after it by mistake, which silently disabled this feature).

In [4]:
from dotenv import load_dotenv
load_dotenv("/Users/russelltamer/Desktop/system 2 RAG/.env")
import os
assert os.environ.get("ANTHROPIC_API_KEY"), "Key not loaded — check .env path"
print("✅ API key loaded safely")


✅ API key loaded safely


In [5]:
import subprocess
subprocess.run(["pip", "install", "-q", "anthropic"])
import anthropic
import os

In [6]:
claude_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

def decontextualize(title, proposition):
    prompt = f"""العنوان: {title}
الجملة المستخرجة من المقال: {proposition}

أعد كتابة الجملة المستخرجة بحيث تكون مستقلة ومفهومة بذاتها، بإضافة السياق الناقص (الموضوع، الفاعل) من العنوان فقط دون إضافة معلومات غير موجودة. أعد فقط الجملة المعاد كتابتها بدون أي شرح إضافي."""
    resp = claude_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text.strip()

**Bucket B** — live news knowledge base (28,800+ propositions).

In [7]:
client = chromadb.PersistentClient(path=CHROMA)

# ── Guards ───────────────────────────────────────────────────────────────────
# Bucket B = scraped news propositions ONLY (independent from Bucket A)
# Sources: original KB (8,376) + scraped v2 (5,808) + SANAD (6,503) = 20,687 total
# AraFacts descriptions removed — they overlap with Bucket A's AraFacts claims
EXPECTED_B_MIN = 20000

try:
    col_b = client.get_collection("bucket_b_propositions")
    if col_b.count() >= EXPECTED_B_MIN:
        print(f"✅ Bucket B already ingested: {col_b.count()} docs — skipping")
    else:
        raise ValueError(f"Incomplete ({col_b.count()} < {EXPECTED_B_MIN})")
except Exception as e:
    print(f"Re-ingesting Bucket B ({e})...")
    try: client.delete_collection("bucket_b_propositions")
    except: pass
    col_b = client.create_collection(
        "bucket_b_propositions", metadata={"hnsw:space": "cosine"})

    # ── Clean propositions from merged news KB (independent evidence source) ─
    df_props = pd.read_csv(PROPS)
    df_props = df_props.drop_duplicates(subset='proposition').reset_index(drop=True)
    print(f"Propositions loaded: {len(df_props)}")
    print(f"Sources: {df_props['source'].value_counts().to_dict()}")

    print("\nEmbedding propositions...")
    vecs_props = get_embeddings_batch(df_props['proposition'].tolist(), batch_size=32)

    BATCH = 500
    for i in range(0, len(df_props), BATCH):
        b_df  = df_props.iloc[i:i+BATCH]
        b_vec = vecs_props[i:i+BATCH]
        col_b.add(
            ids=[f"prop_{j}" for j in range(i, i+len(b_df))],
            embeddings=b_vec.tolist(),
            documents=b_df['proposition'].tolist(),
            metadatas=[{
                "article_id": str(r['article_id']),
                "title":      str(r['title']),
                "source":     str(r['source']),
            } for _, r in b_df.iterrows()]
        )

    print(f"\n✅ Bucket B complete: {col_b.count()} propositions")
    print(f"   Sources: {df_props['source'].value_counts().head(5).to_dict()}")
    print(f"   Note: AraFacts descriptions removed — Bucket B is now independent from Bucket A")

✅ Bucket B already ingested: 72283 docs — skipping


**Bucket A** — verified claims database (AraFacts + Saheeh Masr).

In [8]:
# ── Bucket A: ALL verified claims (TRUE + FALSE + Partly-False) ──────────────
# Renamed collection to reflect correct scope
EXPECTED_A_MIN = 21000   # AraFacts all claims (~18,598) + Saheeh Masr all (~3,019)

LABEL_MAP = {
    "FALSE":        "FALSE",
    "Partly-false": "PARTLY_FALSE",
    "Partly-False": "PARTLY_FALSE",
    "TRUE":         "TRUE",
    "Sarcasm":      "SARCASM",
    "Unverifiable": "UNVERIFIABLE",
}

try:
    col_a = client.get_collection("bucket_a_verified_claims")
    if col_a.count() >= EXPECTED_A_MIN:
        print(f"✅ Bucket A already ingested: {col_a.count()} docs — skipping")
    else:
        raise ValueError(f"Incomplete ({col_a.count()} < {EXPECTED_A_MIN})")
except Exception as e:
    print(f"Re-ingesting Bucket A ({e})...")

    # Drop old collections (both old name and new name if they exist)
    for name in ["bucket_a_fake_claims", "bucket_a_verified_claims"]:
        try: client.delete_collection(name)
        except: pass

    col_a = client.create_collection(
        "bucket_a_verified_claims", metadata={"hnsw:space": "cosine"})

    # ── AraFacts: all 18,598 claims ───────────────────────────────────────────
    df_all_arafacts = df_arafacts[['claim','normalized_label','source','description','date']].copy()
    df_all_arafacts['label'] = df_all_arafacts['normalized_label'].map(LABEL_MAP).fillna("UNKNOWN")
    df_all_arafacts = df_all_arafacts.dropna(subset=['claim']).reset_index(drop=True)
    print(f"AraFacts claims: {len(df_all_arafacts)}")
    print(df_all_arafacts['label'].value_counts().to_string())

    print("\nEmbedding AraFacts claims...")
    vecs_arafacts = get_embeddings_batch(df_all_arafacts['claim'].tolist(), batch_size=32)

    BATCH = 500
    for i in range(0, len(df_all_arafacts), BATCH):
        b_df  = df_all_arafacts.iloc[i:i+BATCH]
        b_vec = vecs_arafacts[i:i+BATCH]
        col_a.add(
            ids=[f"ara_{j}" for j in range(i, i+len(b_df))],
            embeddings=b_vec.tolist(),
            documents=b_df['claim'].tolist(),
            metadatas=[{
                "label":       str(r['label']),
                "source":      str(r['source']),
                "description": str(r['description'])[:400] if pd.notna(r['description']) else "",
                "date":        str(r['date']) if pd.notna(r['date']) else "",
                "dialect":     "",
            } for _, r in b_df.iterrows()]
        )
    print(f"AraFacts ingested: {col_a.count()}")

    # ── Saheeh Masr: all 3,019 claims (TRUE + FALSE) ─────────────────────────
    df_saheeh = pd.read_csv(f"{BASE}/saheeh_masr_claims.csv")
    df_saheeh['claim'] = df_saheeh['claim'].str.replace(
        r'\n?اقرأ التصحيح.*$', '', regex=True).str.strip()
    df_saheeh = df_saheeh.drop_duplicates(subset='claim').reset_index(drop=True)
    df_saheeh = df_saheeh[df_saheeh['claim'].str.len() >= 15].reset_index(drop=True)
    df_saheeh['label'] = df_saheeh['verdict'].map({True: "TRUE", False: "FALSE"})
    print(f"\nSaheeh Masr claims: {len(df_saheeh)}")
    print(df_saheeh['label'].value_counts().to_string())

    print("\nEmbedding Saheeh Masr claims...")
    vecs_saheeh = get_embeddings_batch(df_saheeh['claim'].tolist(), batch_size=32)

    start_id = col_a.count()
    for i in range(0, len(df_saheeh), BATCH):
        b_df  = df_saheeh.iloc[i:i+BATCH]
        b_vec = vecs_saheeh[i:i+BATCH]
        col_a.add(
            ids=[f"saheeh_{start_id + j}" for j in range(i, i+len(b_df))],
            embeddings=b_vec.tolist(),
            documents=b_df['claim'].tolist(),
            metadatas=[{
                "label":       str(r['label']),
                "source":      "saheeh_masr",
                "description": "",
                "date":        str(r['date']) if pd.notna(r['date']) else "",
                "dialect":     "EGY",
            } for _, r in b_df.iterrows()]
        )

    print(f"\n✅ Bucket A complete: {col_a.count()} total verified claims")
    print(f"   Sources: AraFacts ({len(df_all_arafacts)}) + Saheeh Masr ({len(df_saheeh)})")


✅ Bucket A already ingested: 21001 docs — skipping


**Egyptian dialect → MSA normalization**, used to clean queries before retrieval.

In [9]:
# Egyptian dialect (CAI) → MSA word-level mapping
EGY_TO_MSA = {
    # Pronouns
    "انا": "أنا",   "احنا": "نحن",   "انت": "أنت",   "انتو": "أنتم",
    # Negation / existence
    "مش": "ليس",    "مفيش": "لا يوجد",  "معندوش": "ليس لديه",
    # Conjunctions
    "لو": "إذا",    "عشان": "لأن",    "علشان": "لأن",  "بس": "لكن",
    # Adverbs
    "كده": "هكذا",  "كدا": "هكذا",    "اوي": "جداً",
    "دلوقتي": "الآن", "دلوقت": "الآن", "امبارح": "أمس",
    # Interrogatives
    "ايه": "ماذا",  "فين": "أين",     "ازاي": "كيف",
    "امتى": "متى",  "ليه": "لماذا",   "مين": "من",
    # Common verbs (active)
    "جه": "جاء",    "راح": "ذهب",     "قعد": "جلس",    "شاف": "رأى",
    "بيقول": "يقول", "بيعمل": "يفعل", "بيجي": "يأتي",  "بيروح": "يذهب",
    "بيحصل": "يحدث", "بيصير": "يحدث", "بيحدث": "يحدث",
    "بيتكلم": "يتحدث", "بيشتغل": "يعمل", "بيكذب": "يكذب",
    "بيقدر": "يستطيع", "بيحاول": "يحاول",
    "بيشوف": "يرى",  "بنشوف": "نرى",
    "بنقول": "نقول", "بنعمل": "نفعل",
    "هيعمل": "سيفعل", "هيجي": "سيأتي", "هيروح": "سيذهب",
    "عارف": "يعرف",  "عارفة": "تعرف",
    # Passive verbs
    "اتعمل": "صُنع", "اتقال": "قيل",  "اتكلم": "تحدث",
    "اتحط": "وُضع",  "اتبنى": "بُني",
    # Demonstratives (also handled by word-order fix below)
    "ده": "هذا",    "دي": "هذه",     "دول": "هؤلاء",
    # Relative pronoun
    "اللي": "الذي",
    # Misc
    "كمان": "أيضاً", "الاول": "أولاً", "تاني": "ثانياً",
    "كدب": "كذب", "حاجة": "شيء",
}

# ── POS-aware demonstrative reordering (upgrade over position-only heuristic) ──
from camel_tools.disambig.bert import BERTUnfactoredDisambiguator
print("Loading Egyptian Arabic morphological disambiguator...")
_egy_disambiguator = BERTUnfactoredDisambiguator.pretrained(model_name='egy')
print("✅ Disambiguator loaded")

def fix_demonstrative_order(text: str) -> str:
    """
    Move demonstrative (هذا/هذه/هؤلاء) to precede the noun it modifies, using real POS
    tagging instead of a position-only heuristic. Only reorders when the word immediately
    before the demonstrative is actually tagged as a noun -- e.g. correctly leaves
    "تغير مناخي هذا" unchanged since "مناخي" is an adjective, not a noun, avoiding the
    nonsense "هذا مناخي" the old position-only heuristic produced.
    """
    words = text.split()
    demonstratives = {"هذا", "هذه", "هؤلاء"}
    demo_indices = [i for i, w in enumerate(words) if w in demonstratives]
    if not demo_indices:
        return text

    analyses = _egy_disambiguator.disambiguate(words)
    pos_tags = [a.analyses[0].analysis.get('pos') if a.analyses else None for a in analyses]

    result = words[:]
    for idx in demo_indices:
        if idx == 0:
            continue
        if pos_tags[idx - 1] == "noun":
            result[idx - 1], result[idx] = result[idx], result[idx - 1]
        # else: leave it -- not a noun, don't guess
    return " ".join(result)

def normalize_to_msa(text: str) -> str:
    text = normalize_unicode(text)
    text = normalize_alef_ar(text)          # ألف-only — safe; NOT alef_maksura/teh_marbuta
    words = text.split()
    text  = " ".join(EGY_TO_MSA.get(w, w) for w in words)
    text  = fix_demonstrative_order(text)
    return text

def detect_dialect(text: str):
    result = dialect_classifier(text[:512])[0]
    return result['label'], round(result['score'], 3)

def prepare_query(claim: str, verbose: bool = True) -> str:
    """Detect dialect; normalize EGY (CAI) → MSA before embedding."""
    dialect, conf = detect_dialect(claim)
    if verbose:
        print(f"  Dialect: {dialect} (conf={conf})")
    if dialect == 'CAI' and conf > 0.5:
        normalized = normalize_to_msa(claim)
        if verbose:
            print(f"  Original:   {claim}")
            print(f"  Normalized: {normalized}")
        return normalized
    return claim

# Quick smoke test
print(prepare_query("مفيش دليل على الكلام ده اوي"))


Loading Egyptian Arabic morphological disambiguator...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: /Users/russelltamer/.camel_tools/data/disambig_bert_unfactored/egy
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Disambiguator loaded
  Dialect: CAI (conf=1.0)
  Original:   مفيش دليل على الكلام ده اوي
  Normalized: لا يوجد دليل على هذا الكلام جداً
لا يوجد دليل على هذا الكلام جداً


**Arabic stemming + stopwords** — used by BM25 tokenization and the lexical-overlap safety check.

In [10]:
import subprocess
subprocess.run(["pip", "install", "nltk", "-q"])
import nltk
nltk.download('punkt', quiet=True)
from nltk.stem.isri import ISRIStemmer

stemmer = ISRIStemmer()

# Arabic stopwords
ARABIC_STOPWORDS = {
    "من","في","على","إلى","عن","مع","هذا","هذه","هو","هي","هم","هن",
    "أن","إن","كان","كانت","لا","لم","لن","قد","قال","قالت",
    "ما","ماذا","التي","الذي","الذين","وهو","وهي","وكان","ومن",
    "يوجد","توجد","وجود","غير","بعد","قبل","عند","حتى","إذا",
    "لكن","أو","بل","ثم","حيث","كيف","لماذا","متى","أين",
    "كل","بعض","جميع","أي","كما","أيضا","فقط","جدا","وقد",
    "وفي","وعلى","وإلى","وأن","وكانت","وكان",
    "لا","على","عن","في","من","إن","أن","يا","هل","نعم",
}

# Scoped Arabic synonym groups — moved here (from the lexical_overlap cell)
# so bm25_tokenize can use the same normalization at INDEXING time, not just
# in the post-hoc trust check. Without this, a document phrased entirely in
# synonyms of the query (e.g. query says "مياه", document says "الماء") might
# never even be retrieved by BM25 in the first place — normalize_synonyms()
# was previously only applied after retrieval, when deciding whether to trust
# a match, not when deciding what to retrieve.
SYNONYM_GROUPS = [
    {"مياه", "ماء", "الماء", "المياه"},
    {"سعر", "أسعار", "الأسعار", "تسعير", "ثمن", "أثمان"},
    {"رفع", "ارتفاع", "زيادة", "زاد", "ارتفع", "يرتفع"},
    {"خفض", "تخفيض", "انخفاض", "هبوط", "انخفض", "يهبط"},
    {"لتر", "لترات", "ليتر"},
]

_SYNONYM_CANON = {}
for _group in SYNONYM_GROUPS:
    _canon = min(_group)
    for _w in _group:
        _SYNONYM_CANON[_w] = _canon


def normalize_synonyms(tokens) -> set:
    """Map tokens to a canonical form where a scoped synonym mapping exists."""
    return {_SYNONYM_CANON.get(t, t) for t in tokens}


def bm25_tokenize(text: str) -> list:
    """Stopword removal + synonym normalization + ISRI stemming for Arabic BM25 (following Paper 3)."""
    tokens = [w for w in text.split() if w not in ARABIC_STOPWORDS and len(w) > 2]
    tokens = normalize_synonyms(tokens)
    return [stemmer.stem(t) for t in tokens]

# ── Bucket B BM25 index ───────────────────────────────────────────────────────
df_all_b = pd.read_csv(PROPS)
df_all_b = df_all_b.drop_duplicates(subset='proposition').reset_index(drop=True)
df_all_b = df_all_b.rename(columns={'proposition': 'text'})
print(f"Bucket B docs: {len(df_all_b)}")

print("Building BM25 index…")
tokenized_all = [bm25_tokenize(doc) for doc in df_all_b['text'].tolist()]
# k1=1.2, b=0.75 calibrated for Arabic news text (Paper 4 — Hybrid RAG for Islamic QA)
bm25 = BM25Okapi(tokenized_all, k1=1.2, b=0.75)
print(f"✅ BM25 index ready  (k1=1.2, b=0.75, ISRI stemming, {len(df_all_b)} docs)")

Bucket B docs: 77982
Building BM25 index…
✅ BM25 index ready  (k1=1.2, b=0.75, ISRI stemming, 77982 docs)


**Named entity extraction** — boosts retrieval ranking when a claim and evidence share a named entity (person/place/org).

In [11]:
def extract_entities(text: str) -> set:
    """Extract named entities from Arabic text for boosting."""
    try:
        results = ner_pipeline_ar(text[:512])
        return {r['word'].strip() for r in results
                if r['score'] > 0.7 and len(r['word'].strip()) > 2}
    except Exception:
        return set()

def hybrid_search(query_text: str, k: int = 5, k_rrf: int = 60) -> list:
    """
    RRF hybrid retrieval: 1/(k_rrf + rank_e5) + 1/(k_rrf + rank_bm25)
    + NER entity boosting.
    Upgraded from v1 weighted sum (alpha*arabert + (1-alpha)*bm25):
      - RRF uses ranks not raw scores → no normalization needed, no alpha to tune
      - E5-large vectors replace AraBERT → better retrieval alignment
      - NER boost fixes geographic/person entity conflation
    """
    # BM25 ranking
    tokens     = bm25_tokenize(query_text)
    bm25_sc    = bm25.get_scores(tokens)
    bm25_order = np.argsort(bm25_sc)[::-1]
    bm25_ranks = {int(idx): rank + 1 for rank, idx in enumerate(bm25_order)}

    # E5-large ranking from ChromaDB
    qvec    = get_embedding(query_text, prefix="query")
    results = col_b.query(query_embeddings=[qvec.tolist()], n_results=50)
    e5_ranks = {}
    for rank, doc_id in enumerate(results['ids'][0]):
        idx = int(doc_id.split('_')[1])   # "prop_1234" → 1234
        e5_ranks[idx] = rank + 1

    # NER entities from claim for entity overlap boosting
    claim_entities = extract_entities(query_text)

    # Candidate pool: top-50 BM25 ∪ top-50 E5
    all_idx = set(int(i) for i in bm25_order[:50]) | set(e5_ranks.keys())

    candidates = []
    for idx in all_idx:
        r_bm25 = bm25_ranks.get(idx, 100)
        r_e5   = e5_ranks.get(idx, 100)
        rrf    = 1 / (k_rrf + r_bm25) + 1 / (k_rrf + r_e5)

        prop_text = df_all_b.iloc[idx]['text']
        if claim_entities:
            overlap = sum(1 for e in claim_entities if e in prop_text)
            rrf += overlap * 0.005   # small boost per matching entity

        candidates.append({
            "proposition":  prop_text,
            "title":        df_all_b.iloc[idx]['title'],
            "source":       df_all_b.iloc[idx].get('source', ''),
            "hybrid_score": round(rrf, 5),
            "bm25_rank":    r_bm25,
            "e5_rank":      r_e5,
        })

    candidates.sort(key=lambda x: x['hybrid_score'], reverse=True)

    seen, hits = set(), []
    for c in candidates:
        t = c['title']
        if t not in seen:
            seen.add(t)
            hits.append(c)
        if len(hits) >= k:
            break

    return hits

print("✅ hybrid_search ready  (RRF fusion + NER entity boosting)")

✅ hybrid_search ready  (RRF fusion + NER entity boosting)


**Lexical overlap check** — catches cases where two texts look semantically similar (high embedding similarity) but barely share any actual words. Prevents the system from confidently matching a claim to an unrelated stored claim just because they're stylistically alike.

In [12]:
# SYNONYM_GROUPS / normalize_synonyms now live in the bm25_tokenize cell
# above, so the same normalization applies at both retrieval time and here.
def lexical_overlap(text1: str, text2: str) -> float:
    """
    Fraction of shared stemmed content words between two texts (Jaccard overlap),
    with scoped synonym normalization applied first (see SYNONYM_GROUPS above).
    Used to catch cases where E5 reports high semantic similarity but the two
    texts share almost no actual words — a sign E5 may be matching on style/
    adjectives rather than real subject overlap (e.g. "giant crocodile" vs
    "giant water wheel" — both "giant", completely different subject).
    """
    tokens1 = normalize_synonyms(set(bm25_tokenize(text1)))
    tokens2 = normalize_synonyms(set(bm25_tokenize(text2)))
    if not tokens1 or not tokens2:
        return 0.0
    return len(tokens1 & tokens2) / len(tokens1 | tokens2)

print("✅ lexical_overlap ready (scoped synonym normalization applied)")

✅ lexical_overlap ready (scoped synonym normalization applied)


**Cross-encoder reranker** — re-scores Bucket B's retrieved candidates for relevance (F1-calibrated threshold: 2.0).

In [13]:
# ── Cross-encoder Reranker ────────────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "sentence-transformers", "-q"])
from sentence_transformers import CrossEncoder

print("Loading cross-encoder reranker...")
cross_encoder = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")
print("✅ Cross-encoder loaded  (multilingual, trained on MS MARCO)")

def rerank(claim: str, candidates: list, top_k: int = 5) -> list:
    """
    Re-score (claim, proposition) pairs jointly using cross-encoder.
    Unlike cosine similarity (which compares vectors independently),
    the cross-encoder sees both texts together → captures entailment, negation, specificity.
    """
    if not candidates:
        return candidates
    pairs  = [(claim, c['proposition']) for c in candidates]
    scores = cross_encoder.predict(pairs)
    for c, s in zip(candidates, scores):
        c['rerank_score'] = round(float(s), 4)
    reranked = sorted(candidates, key=lambda x: x['rerank_score'], reverse=True)
    return reranked[:top_k]

print("✅ rerank ready")

Loading cross-encoder reranker...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Cross-encoder loaded  (multilingual, trained on MS MARCO)
✅ rerank ready


## 2. Core pipeline — `fact_check_claim()`
This is the main function. Given an Arabic claim, it returns a verdict (TRUE/FALSE/UNVERIFIED), a confidence score, and supporting evidence from both buckets. **This is what System 3 (Youssef) consumes.**

In [14]:
# Decontextualization of displayed Bucket B evidence (Chen et al. 2023) — applied
# only to the top result shown to a user, not the full index (cost/time constraint).
# Cached to disk + wrapped in try/except so a network/API failure NEVER breaks the
# pipeline — falls back to the original (non-decontextualized) text silently.
import json as _json
import os as _os

DECONTEXT_CACHE_PATH = f"{BASE}/decontext_cache.json"
if _os.path.exists(DECONTEXT_CACHE_PATH):
    with open(DECONTEXT_CACHE_PATH, encoding="utf-8") as _f:
        _decontext_cache = _json.load(_f)
else:
    _decontext_cache = {}

def get_decontextualized(title: str, proposition: str) -> str:
    """Returns decontextualized text if available/cacheable, else the original — never raises."""
    key = title + "||" + proposition
    if key in _decontext_cache:
        return _decontext_cache[key]
    try:
        rewritten = decontextualize(title, proposition)
        _decontext_cache[key] = rewritten
        with open(DECONTEXT_CACHE_PATH, "w", encoding="utf-8") as f:
            _json.dump(_decontext_cache, f, ensure_ascii=False, indent=1)
        return rewritten
    except Exception:
        return proposition  # silent fallback — demo never breaks

VERDICT_TO_FINAL = {
    "HIGH_FAKE_MATCH":        "FALSE",
    "HIGH_TRUE_MATCH":        "TRUE",
    "HIGH_PARTIAL_MATCH":     "UNVERIFIED",
    "POSSIBLE_FAKE":          "FALSE",
    "POSSIBLE_TRUE":          "TRUE",
    "POSSIBLE_MATCH":         "UNVERIFIED",
    "POSSIBLE_PARTIAL_MATCH": "UNVERIFIED",
    "EVIDENCE_FOUND":         "UNVERIFIED",
    "LOW_CONFIDENCE":         "UNVERIFIED",
}

HIGH_TO_POSSIBLE = {
    "HIGH_FAKE_MATCH":    "POSSIBLE_FAKE",
    "HIGH_TRUE_MATCH":    "POSSIBLE_TRUE",
    "HIGH_PARTIAL_MATCH": "POSSIBLE_MATCH",
}

def compute_confidence(signal: str, top_a: float = 0.0, top_rerank: float = None) -> float:
    if signal in ("HIGH_FAKE_MATCH", "HIGH_TRUE_MATCH"):
        return round(min(0.97, top_a), 2)
    elif signal == "HIGH_PARTIAL_MATCH":
        return 0.75
    elif signal in ("POSSIBLE_FAKE", "POSSIBLE_TRUE"):
        base = 0.60 + (top_a - 0.84) * 2.5
        return round(min(0.80, max(0.55, base)), 2)
    elif signal in ("POSSIBLE_MATCH", "POSSIBLE_PARTIAL_MATCH"):
        return 0.55
    elif signal == "EVIDENCE_FOUND":
        if top_rerank is not None:
            return round(min(0.80, max(0.35, (top_rerank + 5) / 12)), 2)
        return 0.45
    else:
        return 0.20

def fact_check_claim(claim: str, k: int = 5, verbose: bool = True, classifier_signal: dict = None) -> dict:
    """
    Cascade retrieval:
      Stage 1 — Bucket A (E5-large cosine vs ALL verified claims, labeled)
                HIGH     ≥ 0.86  → near-identical claim, trust directly, skip Bucket B
                POSSIBLE ≥ 0.84  → topically related, ALSO search Bucket B for more evidence
                < 0.84   → fall through to Bucket B
      Stage 2 — Bucket B (RRF E5+BM25 + NER boosting + cross-encoder rerank)
                Reached when: no Bucket A match (verdict=None) OR POSSIBLE-tier match (gather more evidence)
    Output includes final_verdict (TRUE/FALSE/UNVERIFIED) + confidence (0-1).

    classifier_signal: optional dict matching System 1's output schema
        (label, label_id, confidence, probabilities). Passed through
        unmodified into the output JSON under "classifier_signal" — fusion
        with this system's verdict happens downstream in System 3
        (Youssef), per both schemas' stated design ("to be fused with
        the RAG system's independent verdict"). This system does not
        alter its own verdict logic based on it.
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"CLAIM: {claim}")
        print(f"{'='*60}")

    dialect, conf = detect_dialect(claim)
    query_text    = prepare_query(claim, verbose=verbose)
    query_vec     = get_embedding(query_text, prefix="query")

    # ── Stage 1: Bucket A ────────────────────────────────────────────────────
    results_a = col_a.query(query_embeddings=[query_vec.tolist()], n_results=3)
    bucket_a_hits = [
        {
            "claim":      doc,
            "similarity": round(1 - dist, 3),
            "label":      meta.get("label", "UNKNOWN"),
            "source":     meta.get("source", ""),
            "dialect":    meta.get("dialect", ""),
            "debunk":     meta.get("description", "")[:200],
            "suspicious_match": False,  # overwritten below once checked
        }
        for doc, dist, meta in zip(
            results_a['documents'][0],
            results_a['distances'][0],
            results_a['metadatas'][0])
    ]

    # Near-tie disambiguation — generalizes beyond any single case:
    # Chroma ranks by cosine similarity, but a 0.003 gap between #1 and #2
    # is measurement noise, not a meaningful signal that #1 is "more correct."
    # If the top candidates are within NEAR_TIE_MARGIN of each other AND
    # disagree on label (e.g. one FALSE, one TRUE — common when AraFacts has
    # multiple distinct historical claims about the same recurring topic),
    # blindly trusting rank #1 risks an arbitrary, noise-driven verdict.
    # Fix: among near-tied candidates, let lexical overlap with the actual
    # query (word-level grounding) break the tie instead of cosine rank.
    NEAR_TIE_MARGIN = 0.02

    if bucket_a_hits:
        top_score = bucket_a_hits[0]['similarity']
        tied_cluster = [h for h in bucket_a_hits if top_score - h['similarity'] <= NEAR_TIE_MARGIN]
        labels_in_cluster = {h['label'] for h in tied_cluster}

        if len(tied_cluster) > 1 and len(labels_in_cluster) > 1:
            # Conflicting labels among near-tied candidates — use lexical
            # overlap with the query to pick the best-grounded one.
            for h in tied_cluster:
                h['_tiebreak_overlap'] = lexical_overlap(query_text, h['claim'])
            tied_cluster.sort(key=lambda h: h['_tiebreak_overlap'], reverse=True)
            best = tied_cluster[0]

            if verbose:
                print(f"\n[Bucket A — Near-tie conflict detected]")
                for h in tied_cluster:
                    print(f"  sim={h['similarity']:.3f} overlap={h['_tiebreak_overlap']:.3f} [{h['label']}] {h['claim'][:60]}")
                print(f"  → resolved to: [{best['label']}] (highest lexical grounding)")

            # Promote the winning candidate to rank #1 so downstream logic uses it.
            bucket_a_hits.remove(best)
            bucket_a_hits.insert(0, best)

        # Strip the debug-only tiebreak field — internal calculation detail,
        # not part of the documented output schema.
        for h in bucket_a_hits:
            h.pop('_tiebreak_overlap', None)

    top_a     = bucket_a_hits[0]['similarity'] if bucket_a_hits else 0
    top_label = bucket_a_hits[0]['label']      if bucket_a_hits else "UNKNOWN"

    if verbose:
        print(f"\n[Bucket A — Verified Claims DB]")
        for i, h in enumerate(bucket_a_hits[:2], 1):
            print(f"  #{i} ({h['similarity']:.3f}) [{h['label']}] [{h['source']}] {h['claim'][:65]}")

    # ── Verdict signal (Bucket A tier) ───────────────────────────────────────
    if top_a >= 0.86:
        verdict = "HIGH_FAKE_MATCH"    if top_label == "FALSE" else \
                "HIGH_TRUE_MATCH"    if top_label == "TRUE"   else \
                "HIGH_PARTIAL_MATCH"
    elif top_a >= 0.84:
        verdict = "POSSIBLE_FAKE"      if top_label == "FALSE" else \
                "POSSIBLE_TRUE"      if top_label == "TRUE"   else \
                "POSSIBLE_MATCH"
    else:
        verdict = None

    # Lexical sanity check — catch high E5 similarity with near-zero word overlap
    # Runs for verdict-tier matches (>=0.84) AND for the 0.75-0.83 range that
    # System 3 (Youssef) trusts under its LOW_CONFIDENCE override path — without
    # this, that path was reading raw similarity with no sanity-check behind it.
    suspicious_match = False
    if bucket_a_hits and (verdict is not None or top_a >= 0.75):
        overlap = lexical_overlap(query_text, bucket_a_hits[0]['claim'])
        if overlap < 0.20:  # calibrated via F1 sweep on 197 labeled pairs (was 0.15, see reranker_calibration.ipynb)
            suspicious_match = True
            if verbose:
                print(f"  ⚠️  High similarity ({top_a:.3f}) but low lexical overlap ({overlap:.3f}) — distrusting match, forcing Bucket B")

        # Cross-encoder sanity check — catches cases lexical overlap CAN'T:
        # homograph confusion (e.g. "حر" = hot/heat vs. free/floating — same
        # word, different meaning, so overlap looks fine but the topics are
        # unrelated). The cross-encoder reads both texts together and can
        # tell entailment from coincidental word-sharing. Threshold (<0) is
        # a conservative heuristic, NOT F1-calibrated for claim-vs-claim pairs
        # (the existing 2.0 threshold was calibrated for claim-vs-proposition
        # pairs in Bucket B — a different task, reused here only as a sanity
        # flag, not a precise cutoff).
        crossenc_score = rerank(query_text, [{'proposition': bucket_a_hits[0]['claim']}], top_k=1)[0]['rerank_score']
        if crossenc_score < 0:
            suspicious_match = True
            if verbose:
                print(f"  ⚠️  Cross-encoder disagrees with cosine similarity (score={crossenc_score:.3f}) — distrusting match, forcing Bucket B")

        # Persist the flag onto the top hit so it survives into the returned
        # JSON — System 3 can check bucket_a[0]["suspicious_match"] instead of
        # trusting raw similarity alone in its LOW_CONFIDENCE override path.
        bucket_a_hits[0]['suspicious_match'] = suspicious_match

    # Only stop early for HIGH-tier AND not suspicious
    if verdict is not None and verdict.startswith("HIGH") and not suspicious_match:
        final_verdict = VERDICT_TO_FINAL[verdict]
        confidence    = compute_confidence(verdict, top_a=top_a)
        if verbose:
            print(f"\n[Verdict Signal]: {verdict}")
            print(f"[Final Verdict]:  {final_verdict}  (confidence={confidence})")
            print(f"  Bucket B skipped (HIGH confidence)")
        return {
            "claim": claim, "dialect": dialect, "dialect_confidence": conf,
            "query_used": query_text, "verdict_signal": verdict,
            "final_verdict": final_verdict, "confidence": confidence,
            "bucket_a": bucket_a_hits, "bucket_b": [], "bucket_b_searched": False,
            "classifier_signal": classifier_signal,
        }

    # FIXED: explicit mapping instead of blind string replace
    if suspicious_match and verdict in HIGH_TO_POSSIBLE:
        verdict = HIGH_TO_POSSIBLE[verdict]

    # ── Stage 2: Bucket B ────────────────────────────────────────────────────
    if verbose:
        reason = f"POSSIBLE signal ({verdict})" if verdict else f"weak Bucket A ({top_a:.3f})"
        print(f"  {reason} → searching Bucket B…")

    bucket_b_raw  = hybrid_search(query_text, k=20)
    bucket_b_hits = rerank(query_text, bucket_b_raw, top_k=k)
    top_rerank    = bucket_b_hits[0]['rerank_score'] if bucket_b_hits else None

    # Threshold below which Bucket B's "top hit" is noise, not a real candidate —
    # distinct from EVIDENCE_FOUND's >2 cutoff. Below this, Bucket B is silent
    # (no relevant evidence exists, e.g. a KB coverage gap), not contradicting.
    NO_EVIDENCE_RERANK = -2.0

    if verdict is not None:
        final_signal = verdict
        confidence   = compute_confidence(verdict, top_a=top_a)
        # FIX (bug #4): don't blindly trust a carried-over POSSIBLE signal —
        # apply the same lexical-overlap entity-confusion check (crocodile
        # case, Sciavolino et al. 2021) to Bucket B's own top hit before
        # accepting it. Without this, a vague claim like "مشروع دبي القادم"
        # that shares only a place name with the evidence gets confidently
        # mismatched.
        #
        # BUT: only apply this check when Bucket B actually returned a plausible
        # candidate (rerank > NO_EVIDENCE_RERANK). A deeply negative rerank score
        # means Bucket B found nothing relevant at all (KB coverage gap) — that's
        # silence, not contradiction, and silence should not override a
        # reasonably-grounded Bucket A match. Conflating "no evidence" with
        # "contradicting evidence" was downgrading correct Bucket A verdicts
        # whenever the KB simply didn't cover the topic (e.g. metro ticket
        # pricing — Bucket B has metro construction news, zero pricing coverage).
        if bucket_b_hits and top_rerank is not None and top_rerank > NO_EVIDENCE_RERANK:
            b_overlap = lexical_overlap(query_text, bucket_b_hits[0]['proposition'])
            if b_overlap < 0.20:  # calibrated threshold (was 0.15)
                if verbose:
                    print(f"  ⚠️  Bucket B top hit shares almost no words with claim (overlap={b_overlap:.3f}) — downgrading to LOW_CONFIDENCE")
                final_signal = "LOW_CONFIDENCE"
                confidence   = compute_confidence(final_signal, top_rerank=top_rerank)
        elif verbose:
            print(f"  Bucket B found no real candidates (top_rerank={top_rerank}) — sticking with Bucket A verdict ({verdict})")
    else:
        final_signal = "EVIDENCE_FOUND" if (top_rerank is not None and top_rerank > 2) else "LOW_CONFIDENCE"  # calibrated threshold (F1 0.471 -> 0.667)
        confidence   = compute_confidence(final_signal, top_rerank=top_rerank)

    final_verdict = VERDICT_TO_FINAL[final_signal]

    # Attach a decontextualized version of the top evidence for display (cached + fail-safe)
    if bucket_b_hits:
        bucket_b_hits[0]['proposition_display'] = get_decontextualized(
            bucket_b_hits[0]['title'], bucket_b_hits[0]['proposition']
        )

    if verbose:
        print(f"\n[Bucket B — RRF + Rerank]")
        for i, h in enumerate(bucket_b_hits[:3], 1):
            print(f"  #{i} (rerank={h.get('rerank_score','?'):.3f}) {h['proposition'][:70]}")
            print(f"       ↳ {h['title'][:60]}")
        print(f"\n[Verdict Signal]: {final_signal}")
        print(f"[Final Verdict]:  {final_verdict}  (confidence={confidence})")

    return {
        "claim":              claim,
        "dialect":            dialect,
        "dialect_confidence": conf,
        "query_used":         query_text,
        "verdict_signal":     final_signal,
        "final_verdict":      final_verdict,
        "confidence":         confidence,
        "bucket_a":           bucket_a_hits,
        "bucket_b":           bucket_b_hits,
        "bucket_b_searched":  True,
        "classifier_signal":  classifier_signal,
    }

print("✅ fact_check_claim ready (+ cached, fail-safe decontextualization of displayed evidence)")

✅ fact_check_claim ready (+ cached, fail-safe decontextualization of displayed evidence)


Quick sanity check — run one claim to confirm everything loaded correctly.

In [15]:
fact_check_claim("لقاح أسترازينيكا يسبب جلطات دموية")


CLAIM: لقاح أسترازينيكا يسبب جلطات دموية
  Dialect: TUN (conf=0.879)

[Bucket A — Near-tie conflict detected]
  sim=0.872 overlap=0.115 [TRUE] ⚠️ نشر موقع قناة العربية، خبرا عن تعليق دولة النمسا، استخدام
  sim=0.857 overlap=0.062 [FALSE] صورة قائمة تحذيرية وضعتها هيئة تنظيم المنتجات الصحية في إيرل
  → resolved to: [TRUE] (highest lexical grounding)

[Bucket A — Verified Claims DB]
  #1 (0.872) [TRUE] [saheeh_masr] ⚠️ نشر موقع قناة العربية، خبرا عن تعليق دولة النمسا، استخدام لقاح
  #2 (0.857) [FALSE] [Misbar] صورة قائمة تحذيرية وضعتها هيئة تنظيم المنتجات الصحية في إيرلندا، 
  ⚠️  High similarity (0.872) but low lexical overlap (0.115) — distrusting match, forcing Bucket B
  POSSIBLE signal (POSSIBLE_TRUE) → searching Bucket B…
  Bucket B found no real candidates (top_rerank=-2.235) — sticking with Bucket A verdict (POSSIBLE_TRUE)

[Bucket B — RRF + Rerank]
  #1 (rerank=-2.235) ويشير الباحثون إلى تحديات علمية كبيرة تواجه هذه التقنيات، من أبرزها تك
       ↳ تجميد الجثث.. هل يصبح الموت مر

{'claim': 'لقاح أسترازينيكا يسبب جلطات دموية',
 'dialect': 'TUN',
 'dialect_confidence': 0.879,
 'query_used': 'لقاح أسترازينيكا يسبب جلطات دموية',
 'verdict_signal': 'POSSIBLE_TRUE',
 'final_verdict': 'TRUE',
 'confidence': 0.68,
 'bucket_a': [{'claim': '⚠️ نشر موقع قناة العربية، خبرا عن تعليق دولة النمسا، استخدام لقاح أسترازينيكا المطور من جامعة أكسفورد، بعد وفاة إحدى المواطنات وإصابة أخرى بتجلط في الدم عقب تطعيمهما باللقاح.',
   'similarity': 0.872,
   'label': 'TRUE',
   'source': 'saheeh_masr',
   'dialect': 'EGY',
   'debunk': '',
   'suspicious_match': True},
  {'claim': 'صورة قائمة تحذيرية وضعتها هيئة تنظيم المنتجات الصحية في إيرلندا، للتحذير من عواقب لقاح كورونا والتي تشمل الصداع، واضطرابات الدورة الشهرية، وشلل الوجه النصفي، وجلطات الدم ، والنوبات القلبية، والسكتات الدماغية، والموت المفاجئ.',
   'similarity': 0.857,
   'label': 'FALSE',
   'source': 'Misbar',
   'dialect': '',
   'debunk': 'تداولت حسابات وصفحات على موقع التواصل الاجتماعي فيسبوك، حديثًا ومنذ أشهر، صورةً قالت إن

**Source links** — maps article titles to their original URLs, shown in the demo UI.

In [16]:
_truth_links = pd.read_csv(f"{BASE}/Bucket_B_Truth_CLEAN_FINAL.csv")
TITLE_TO_LINK = dict(zip(_truth_links['title'], _truth_links['link']))
print(f"✅ Loaded {len(TITLE_TO_LINK)} title->link mappings")

✅ Loaded 6144 title->link mappings


## 2. Load System 1 (Sarah's classifier) and System 3 (Youssef's Verdict Engine)
Both load real trained models — Sarah's from Hugging Face, Youssef's NLI model (`xlm-roberta-large-xnli`, ~2.8GB) on first run.

In [17]:
import sys

sys.path.insert(0, "/Users/russelltamer/Desktop/system 1")
from arabic_fake_news_classifier import ArabicFakeNewsClassifier

sys.path.insert(0, "/Users/russelltamer/Desktop/system 2 RAG/system2b")
from agents.verdict_engine import VerdictEngine

print("Loading System 1 (Sarah's classifier, v3 MARBERTv2+TAPT)...")
sarah_clf = ArabicFakeNewsClassifier("sarahhh33/arabic-fake-news-marbertv2")
print(f"✅ System 1 loaded — {sarah_clf.model_version}, threshold={sarah_clf.threshold}\n")

print("Loading System 3 (Youssef's Verdict Engine — includes NLI model)...")
verdict_engine = VerdictEngine()
print("✅ System 3 loaded")


def adapt_classifier_output(new_output: dict) -> dict:
    """
    Sarah's classifier was upgraded (v1 -> v3, MARBERTv2+TAPT) with a new
    output schema AND a flipped label convention:
        old: {"label": "fake"/"real", "label_id": 0=real/1=fake}
        new: {"prediction": "Fake"/"Real", "label": 0=FAKE/1=REAL}
    Without this adapter, Youssef's verdict_engine.py (which reads
    classifier_signal["label"] expecting the lowercase string "fake"/"real")
    would silently misread every signal, since the new "label" field is now
    an integer. This adapts v3's output back into the original schema so
    neither this pipeline nor System 3 needs to change just because System 1's
    internal model changed.
    """
    label_str = new_output["prediction"].lower()       # "Fake" -> "fake"
    old_label_id = 1 if label_str == "fake" else 0      # OLD convention: 0=real, 1=fake
    return {
        "label": label_str,
        "label_id": old_label_id,
        "confidence": new_output["confidence"],
        "probabilities": new_output["probabilities"],
    }

Loading System 1 (Sarah's classifier, v3 MARBERTv2+TAPT)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ System 1 loaded — v3-marbertv2-tapt, threshold=0.34

Loading System 3 (Youssef's Verdict Engine — includes NLI model)...
[NLI] Loading model... (one-time, ~2.8GB)
[NLI] Using device: cpu


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[NLI] Model ready.
✅ System 3 loaded


## 3. Combined pipeline
`full_pipeline(claim)` — the actual end-to-end call: classifies, retrieves evidence, verifies, and fuses all three signals into one final verdict.

In [18]:
def full_pipeline(claim: str, verbose: bool = False) -> dict:
    """
    Runs a claim through all three systems and returns a combined result.

    Returns a dict with:
        - classifier_signal : System 1's output, adapted to the original schema
        - rag_output         : System 2's full output (verdict, evidence, etc.)
        - final              : System 3's FinalOutput (fused verdict, confidence, reason)
    """
    raw_signal = sarah_clf.predict(claim)
    classifier_signal = adapt_classifier_output(raw_signal)
    rag_output = fact_check_claim(claim, classifier_signal=classifier_signal, verbose=verbose)
    final = verdict_engine.decide(rag_output)

    return {
        "claim": claim,
        "classifier_signal": classifier_signal,
        "rag_output": rag_output,
        "final": final,
    }


def print_full_result(result: dict) -> None:
    """Pretty-print a full_pipeline() result for manual inspection."""
    print("=" * 70)
    print(f"CLAIM: {result['claim']}")
    print("=" * 70)
    cs = result["classifier_signal"]
    print(f"[System 1 — Sarah]   {cs['label']} ({cs['confidence']:.2f})")
    rag = result["rag_output"]
    print(f"[System 2 — Russell] {rag['final_verdict']} ({rag['confidence']:.2f}) | signal={rag['verdict_signal']}")
    final = result["final"]
    print(f"[System 3 — Youssef] {final['final_verdict']} ({final['confidence']:.2f})")
    print(f"  Reason: {final['reason']}")
    print(f"  Classifier fusion: {final['classifier_fusion']}")
    print("=" * 70)

Quick sanity check — run one claim through all three systems.

In [19]:
result = full_pipeline("السيسي باع سيناء للإسرائيليين")
print_full_result(result)

CLAIM: السيسي باع سيناء للإسرائيليين
[System 1 — Sarah]   fake (0.99)
[System 2 — Russell] FALSE (0.71) | signal=POSSIBLE_FAKE
[System 3 — Youssef] TRUE (0.86)
  Reason: [Rule 1 — Solid Bucket A] Russell found solid fact-checked evidence in known-fakes database (similarity 0.865, source: saheeh_masr). System 1 classifier disagreement is overridden because fact-checked evidence outranks classifier prediction. The fact-checked evidence from saheeh_masr strongly indicates the claim is TRUE (confirmed by fact-checkers).
  Classifier fusion: {'used': False, 'label': 'fake', 'confidence': 0.9855, 'effect': 'ignored'}


## 4. Full-system Gradio interface
Type a claim, see all three systems' opinions and the final fused verdict.

In [20]:
import gradio as gr

def run_full_demo(claim):
    if not claim.strip():
        return "<p style='color:#9A8070; text-align:center; font-family:system-ui; direction:rtl; padding:20px;'>أدخل ادعاءً للتحقق منه</p>"

    result = full_pipeline(claim, verbose=False)
    cs, rag, final = result["classifier_signal"], result["rag_output"], result["final"]

    s1_pred = "FALSE" if cs["label"] == "fake" else "TRUE"
    s1_conf = int(cs["confidence"] * 100)
    s2_pred = rag["final_verdict"]
    s2_conf = int(rag["confidence"] * 100)

    verdict_ar = {"TRUE": "صحيح", "FALSE": "كاذب", "UNVERIFIED": "غير مؤكد"}.get(final["final_verdict"], final["final_verdict"])
    verdict_color = {"TRUE": "#1A5C1A", "FALSE": "#7A1515", "UNVERIFIED": "#7A5C00"}.get(final["final_verdict"], "#1A1A1A")
    conf_pct = int(final["confidence"] * 100)

    stance_breakdown = final.get("stance_breakdown", [])

    # ── Merge both buckets, sort by relevance, take top 3 ──
    all_evidence = []

    if stance_breakdown:
        stance_ar = {"SUPPORTS": "يدعم الادعاء", "REFUTES": "يفند الادعاء", "NEUTRAL": "محايد"}
        qualified_b = [p for p in rag.get("bucket_b", []) if p.get("rerank_score", -999) > 2.0]
        for i, prop in enumerate(qualified_b):
            sd = stance_breakdown[i] if i < len(stance_breakdown) else None
            all_evidence.append({
                "text":   prop.get("proposition_display") or prop.get("proposition", ""),
                "debunk": "",
                "source": prop.get("source", ""),
                "tag":    stance_ar.get(sd["stance"], "") if sd else "",
                "link":   TITLE_TO_LINK.get(prop.get("title", ""), ""),
                "score":  prop.get("rerank_score", 0),
                "bucket": "أخبار حية",
            })
    else:
        for hit in rag.get("bucket_a", []):
            all_evidence.append({
                "text":   hit.get("claim", ""),
                "debunk": hit.get("debunk", "") or "",
                "source": hit.get("source", ""),
                "tag":    hit.get("label", ""),
                "link":   "",
                "score":  hit.get("similarity", 0),
                "bucket": "قاعدة التحقق",
            })

    top_evidence = sorted(all_evidence, key=lambda x: x["score"], reverse=True)[:3]

    html = f"""
    <div style="background:#EDE8DC; font-family:'Georgia', serif; direction:rtl; text-align:right; max-width:760px; margin:0 auto; padding:24px;">

      <div style="background:#E4DDD0; border:1px solid #C8C0B0; border-radius:4px; padding:12px 14px; margin-bottom:18px;">
        <div style="font-size:10px; color:#7A1515; font-family:system-ui; letter-spacing:0.1em; margin-bottom:6px;">الادعاء</div>
        <div style="color:#1A1A1A; font-size:15px; font-family:system-ui; line-height:1.6;">{claim}</div>
      </div>

      <div style="border-top:2px solid #1A1A1A; padding-top:16px; margin-bottom:16px;">
        <div style="font-size:9px; color:#7A1515; letter-spacing:0.15em; font-family:system-ui; margin-bottom:8px;">— القرار النهائي</div>
        <div style="font-size:56px; font-weight:900; color:{verdict_color}; line-height:1; font-family:'Georgia', serif; margin-bottom:10px;">{verdict_ar}</div>
        <div style="display:flex; align-items:center; gap:12px;">
          <div style="flex:1; background:#C8C0B0; border-radius:2px; height:6px;">
            <div style="background:{verdict_color}; width:{conf_pct}%; height:6px; border-radius:2px;"></div>
          </div>
          <span style="font-size:12px; color:#5A5248; font-family:system-ui; white-space:nowrap;">نسبة الثقة · {conf_pct}%</span>
        </div>
      </div>

      <div style="display:flex; gap:0; margin-bottom:18px; border-top:1px solid #C8C0B0; border-bottom:1px solid #C8C0B0;">
        <div style="flex:1; padding:12px 12px 12px 0; border-left:1px solid #C8C0B0;">
          <div style="font-size:9px; color:#7A1515; letter-spacing:0.12em; font-family:system-ui; margin-bottom:4px;">SYSTEM 1 — CLASSIFIER</div>
          <div style="font-size:16px; font-weight:700; color:#1A1A1A; font-family:system-ui;">{s1_pred}</div>
          <div style="font-size:12px; color:#5A5248; font-family:system-ui;">{s1_conf}% confidence</div>
        </div>
        <div style="flex:1; padding:12px 12px 12px 0;">
          <div style="font-size:9px; color:#7A1515; letter-spacing:0.12em; font-family:system-ui; margin-bottom:4px;">SYSTEM 2 — RAG</div>
          <div style="font-size:16px; font-weight:700; color:#1A1A1A; font-family:system-ui;">{s2_pred}</div>
          <div style="font-size:12px; color:#5A5248; font-family:system-ui;">{s2_conf}% confidence</div>
        </div>
      </div>

      <div style="border-top:2px solid #1A1A1A; padding-top:14px;">
        <div style="font-size:9px; color:#7A1515; letter-spacing:0.15em; font-family:system-ui; margin-bottom:12px;">— الأدلة المستخدمة في القرار</div>
    """

    if top_evidence:
        for ev in top_evidence:
            tag = ev["tag"]
            tag_color = (
                "#7A1515" if tag in ("FALSE", "PARTLY_FALSE", "يفند الادعاء")
                else "#1A5C1A" if tag in ("TRUE", "يدعم الادعاء")
                else "#5A5248"
            )
            html += f"""
        <div style="border-top:1px solid #C8C0B0; padding:12px 0;">
          <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;">
            <div style="display:flex; gap:8px; align-items:center;">
              {"<span style='font-size:10px; color:" + tag_color + "; border:1px solid " + tag_color + "; padding:2px 10px; font-family:system-ui; letter-spacing:0.06em;'>" + tag + "</span>" if tag else ""}
              <span style="font-size:10px; color:#9A8A80; font-family:system-ui;">score: {round(ev['score'], 2)}</span>
            </div>
            <span style="font-size:11px; color:#5A5248; font-family:system-ui;">{ev['source']}</span>
          </div>
          <div style="font-size:14px; color:#1A1A1A; line-height:1.7; font-family:system-ui; margin-bottom:6px;">{ev['text'][:280]}</div>
          {"<div style='background:#E4DDD0; border-right:3px solid #7A1515; padding:10px 14px; font-size:12px; color:#5A5248; font-family:system-ui; line-height:1.7; margin-bottom:6px;'>" + ev['debunk'] + "</div>" if ev['debunk'] else ""}
          {"<a href='" + ev['link'] + "' target='_blank' style='font-size:11px; color:#7A1515; font-family:system-ui; text-decoration:none;'>← رابط المصدر</a>" if ev['link'] else "<span style='font-size:10px; color:#9A8A80; font-family:system-ui;'>لا يوجد رابط</span>"}
        </div>"""
    else:
        html += """
        <div style="padding:14px 0; text-align:center;">
          <div style="font-size:13px; color:#5A5248; font-family:system-ui;">لم يتم العثور على أدلة كافية</div>
        </div>"""

    html += """
      </div>
      <div style="border-top:1px solid #C8C0B0; margin-top:20px; padding-top:10px; text-align:center;">
        <span style="font-size:9px; color:#5A5248; font-family:system-ui; letter-spacing:0.08em;">Russell Tamer · Sarah Abo Elazm · Youssef Khedr</span>
      </div>
    </div>"""

    return html


def show_loading():
    return """
    <div style="background:#EDE8DC; padding:40px; text-align:center; font-family:system-ui; direction:rtl; border-radius:4px;">
      <div style="font-size:12px; color:#7A1515; letter-spacing:0.15em;">جاري التحقق من الادعاء...</div>
    </div>
    """


with gr.Blocks(
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.red,
        neutral_hue=gr.themes.colors.stone,
    ),
    css="""
        .gradio-container { background: #EDE8DC !important; }
        button.primary { background: #7A1515 !important; border-color: #7A1515 !important; color: #EDE8DC !important; font-weight: 700 !important; letter-spacing: 0.08em !important; font-family: system-ui !important; }
        button.primary:hover { background: #5C0F0F !important; border-color: #5C0F0F !important; }
        textarea, input { background: #E4DDD0 !important; border-color: #C8C0B0 !important; color: #1A1A1A !important; font-family: system-ui !important; }
        label { color: #5A5248 !important; font-family: system-ui !important; font-size: 12px !important; }
        .examples { background: #E4DDD0 !important; }
    """,
    title="Arabic Fake News Detection & Fact Verification"
) as demo:

    gr.HTML("""
        <div style="background:#EDE8DC; font-family:'Georgia', serif; text-align:center; padding:20px 0 16px; border-bottom:3px double #1A1A1A; margin-bottom:16px;">
            <div style="font-size:11px; color:#7A1515; letter-spacing:0.18em; font-family:system-ui; margin-bottom:6px;">A HYBRID AI FRAMEWORK FOR</div>
            <div style="font-size:30px; font-weight:900; color:#1A1A1A; line-height:1.2; font-family:'Georgia', serif;">Arabic Fake News Detection</div>
            <div style="font-size:12px; color:#5A5248; letter-spacing:0.1em; font-family:system-ui; margin-top:4px;">& FACT VERIFICATION</div>
        </div>
    """)

    with gr.Column():
        claim_input = gr.Textbox(
            label="أدخل الادعاء بالعربية",
            lines=3,
            rtl=True,
            placeholder="اكتب الادعاء هنا..."
        )
        submit_btn = gr.Button("يلا اتأكد", variant="primary")

    output = gr.HTML()

    submit_btn.click(fn=show_loading, outputs=output).then(fn=run_full_demo, inputs=claim_input, outputs=output)
    claim_input.submit(fn=show_loading, outputs=output).then(fn=run_full_demo, inputs=claim_input, outputs=output)

demo.launch(share=True)

/var/folders/c0/qtdfk24n7fvg27_6xny54wv80000gn/T/ipykernel_14058/4056735709.py:133: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://3f7ab4a3b83487d934.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 5. Accuracy evaluation — full 3-system pipeline
Uses the 50 known claims + their paraphrases (`para_data.json`) — leakage-free since the paraphrase text was never indexed, and ground truth is known. Runs each paraphrase through **all three systems** and checks the final fused verdict against the true label.

In [21]:
import json

with open(f"{BASE}/para_data.json", encoding="utf-8") as f:
    para_data = json.load(f)

eval_rows = []
for entry in para_data:
    original = entry["original"]
    match = df_arafacts[df_arafacts["claim"] == original]
    if match.empty:
        continue
    true_label = LABEL_MAP.get(match.iloc[0]["normalized_label"], match.iloc[0]["normalized_label"])
    true_verdict = "TRUE" if true_label == "TRUE" else ("FALSE" if true_label == "FALSE" else "UNVERIFIED")

    for level, para_text in entry["paraphrases"].items():
        result = full_pipeline(para_text, verbose=False)
        rag = result["rag_output"]
        final = result["final"]

        qualified_count = sum(1 for p in rag.get("bucket_b", []) if p.get("rerank_score", -999) > 2.0)

        eval_rows.append({
            "level": level,
            "original": original[:50],
            "paraphrase": para_text[:50],
            "true_verdict": true_verdict,
            "rag_verdict_signal": rag["verdict_signal"],
            "rag_final_verdict": rag["final_verdict"],       # System 2's own verdict, BEFORE fusion
            "rag_correct": rag["final_verdict"] == true_verdict,
            "qualified_bucket_b_count": qualified_count,       # how many props passed rerank>2.0
            "predicted_verdict": final["final_verdict"],       # System 3's fused final verdict
            "correct": final["final_verdict"] == true_verdict,
            "confidence": final["confidence"],
            "fusion_effect": final["classifier_fusion"]["effect"],
        })

df_full_eval = pd.DataFrame(eval_rows)
print(f"Evaluated {len(df_full_eval)} paraphrases through the full 3-system pipeline\n")

print("── Accuracy by paraphrase intensity ──")
print(df_full_eval.groupby("level").agg(
    accuracy=("correct", "mean"),
    mean_confidence=("confidence", "mean"),
    n=("correct", "size"),
).round(3))
print(f"\nOverall accuracy (full pipeline, after fusion): {df_full_eval['correct'].mean():.1%}")

# ── Diagnosis: was System 2 (RAG) already correct before fusion touched it? ──
print("\n── System 2's own accuracy, BEFORE System 3 fusion ──")
print(f"RAG-only accuracy: {df_full_eval['rag_correct'].mean():.1%}")

print("\n── Cases where RAG was correct but fusion changed the answer (made it WRONG) ──")
flipped_to_wrong = df_full_eval[(df_full_eval["rag_correct"]) & (~df_full_eval["correct"])]
print(f"Count: {len(flipped_to_wrong)} / {len(df_full_eval)} ({len(flipped_to_wrong)/len(df_full_eval):.1%})")
print(flipped_to_wrong["fusion_effect"].value_counts())

print("\n── Breakdown: fusion_effect vs whether Bucket B had qualifying evidence ──")
print(df_full_eval.groupby(["fusion_effect", df_full_eval["qualified_bucket_b_count"] > 0]).size())

Evaluated 200 paraphrases through the full 3-system pipeline

── Accuracy by paraphrase intensity ──
         accuracy  mean_confidence   n
level                                 
dialect      0.96            0.895  50
heavy        0.84            0.851  50
light        0.98            0.912  50
medium       0.82            0.866  50

Overall accuracy (full pipeline, after fusion): 90.0%

── System 2's own accuracy, BEFORE System 3 fusion ──
RAG-only accuracy: 93.5%

── Cases where RAG was correct but fusion changed the answer (made it WRONG) ──
Count: 16 / 200 (8.0%)
fusion_effect
ignored    15
absent      1
Name: count, dtype: int64

── Breakdown: fusion_effect vs whether Bucket B had qualifying evidence ──
fusion_effect  qualified_bucket_b_count
absent         False                        16
ignored        False                       158
reinforced     False                        26
dtype: int64


## 6. Held-out leakage check — VERA-Arabic dataset
Reproduces Sarah's exact train/test split (same seed, same cap) on VERA-Arabic, then samples only from the genuinely unseen portion — claims neither System 1 nor System 2 encountered during training/ingestion — and runs them through the full pipeline.

In [22]:
from sklearn.model_selection import train_test_split

vera = pd.read_excel(f"{BASE}/VERAARAB_dataset.xlsx")
vera['text'] = vera['text_arabic'].astype(str)
vera['binary_label'] = vera['label'].astype(int)   # 1 = Fake, 0 = Real
vera = vera.drop_duplicates(subset=['text']).reset_index(drop=True)

# Reproduce Sarah's exact split — same seed, same cap
vera_capped, vera_unused = train_test_split(
    vera, train_size=7000, stratify=vera['binary_label'], random_state=42
)
print(f"Reproduced split — used in training: {len(vera_capped)}, genuinely unused: {len(vera_unused)}")

# Now sample from the UNUSED portion only — unseen by both System 1 and System 2
sample = vera_unused.sample(20, random_state=7).reset_index(drop=True)

correct = 0
rows = []
for _, row in sample.iterrows():
    claim = row['text']
    true_label = "FALSE" if row['binary_label'] == 1 else "TRUE"   # 1=Fake→FALSE, 0=Real→TRUE
    result = full_pipeline(claim)
    predicted = result['final']['final_verdict']
    correct += (predicted == true_label)
    rows.append({"claim": claim[:50], "true_label": true_label, "predicted": predicted,
                 "bucket_a_hits": len(result['rag_output'].get('bucket_a', []))})

print(f"Accuracy on {len(sample)} genuinely unseen claims: {correct/len(sample):.1%}")
pd.DataFrame(rows)

Reproduced split — used in training: 7000, genuinely unused: 10638
Accuracy on 20 genuinely unseen claims: 35.0%


,claim,true_label,predicted,bucket_a_hits
0,بدأ الوجع الكبير مستوطن خلاص حماس.. كفى إلى أي...,FALSE,FALSE,3
1,هتافات أبناء حضرموت للمرتزق عيدروس الزبيدي بال...,FALSE,FALSE,3
2,7 الاتحاد نونو سانتوس الاهلي مورينيو مب طبي...,FALSE,FALSE,3
3,حقيقة صورة طابور الحليب في الجزائر التي حظيت ب...,TRUE,FALSE,3
4,إهانة جديدة للرئيس الفرنسي إيمانويل ماكرون رب...,FALSE,UNVERIFIED,3
5,النصر السعودي سيواجه الاتحاد السكندري يوم ١ ما...,FALSE,FALSE,3
6,الأسطورة الايفواريه ديديه دروجبا يعتنق الإسلام...,FALSE,TRUE,3
7,صالة قرطبة ردم جد و اللابس تشيرت ابيض يمثلني,TRUE,FALSE,3
8,وجدوافي انقاض تركيا مليون و200ألف دولاردون ايج...,FALSE,FALSE,3
9,فكره لملعب تنس في العلا,TRUE,FALSE,3
